In [1]:
max_samples = 100
n_samples_per_context = 1  # Number of rows per context

In [2]:
output_file_dataset_filterd_full = "dataset_filtered_full.xlsx"
output_file_dataset_filterd_dropped = "dataset_filtered_dropped.xlsx"

output_file_queries_in_dataset = f"queries_{max_samples}_uniques_in_dataset.xlsx"
output_file_queries_similar_dataset_without_context = f"queries_{max_samples}_uniques_similar_dataset_without_context.xlsx"
output_file_queries_similar_dataset_within_context = f"queries_{max_samples}_uniques_similar_dataset_within_context.xlsx"

In [3]:
import pandas as pd
from datasets import load_dataset

c:\Users\USER\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import openai
import json
import os

# Load OpenAI API key
key_path = './config.json'
with open(key_path, "r") as file:
    config = json.load(file)
os.environ["OPENAI_API_KEY"] = config["openai_api_key"]

openai.api_key = config["openai_api_key"]

In [5]:
def is_not_empty_answer(example):
    return bool(example['answers']['text'])

def is_having_no_common_element(list1, list2):
    """Checks if two lists have no common element."""
    set1 = set(list1)
    set2 = set(list2)
    return not set1.intersection(set2)

def call_chatgpt(user_message: dict,
        sys_message={"role": "system", "content": "You are a helpful assistant that responds in Vietnamese."},
        model="gpt-4o-mini",
        max_tokens=1000,
        temperature=0):
    response = openai.chat.completions.create(
        model=model,  # Use the appropriate model
        messages=[sys_message, user_message],
        max_tokens=max_tokens,  # Further reduced to ensure brevity
        temperature=temperature  # Keep at 0 for precision
    )

    # Get the content of the response
    response_content = response.choices[0].message.content.strip()

    # Return the extracted answer directly
    return response_content

def generate_similar_question(question: str, context: str=None):
    sys_message = {"role": "system", "content": "You are a helpful assistant that responds in Vietnamese."}
    
    if context is None:
        user_content = (f"Question: {question}\nPlease only extract exactly the question (after 'Question: ') and reponse only 1 SIMILAR question to the question with the following format, no more word:\nSimilar question: <similar question>\n")
    else:
        user_content = (f"Question: {question}\nContext: {context}\nPlease extract the question (after 'Question: '), the context (after 'Context: ') and reponse only 1 SIMILAR question to the question with the context with the following format, no more word:\nSimilar question: <similar question>\n")
    user_message = {"role": "user", "content": user_content}
    
    chatgpt_response = call_chatgpt(user_message, sys_message)
    print("ChatGPT: ", chatgpt_response)
    
    return chatgpt_response

def get_answer_from_response(response: str, prefix_str: str):
    return response[len(prefix_str):]

def get_values_in_object_lst(object_lst: list, attribute_str: str):
    return [obj[attribute_str].tolist()[0] for obj in object_lst]

def get_answer_texts_in_object_lst(object_lst: list):
    return [obj['answers'].tolist()[0]['text'].tolist()[0] for obj in object_lst]

def save_lists_dict_to_excel(filename, lists_dict: dict):
    """
    Saves multiple lists dictionary to an Excel file with column names.
    Format dict: {"column_name": list}
    """
    # Create a pandas DataFrame from the dictionary of lists
    df = pd.DataFrame(lists_dict)

    # Save the DataFrame to an Excel file
    df.to_excel(filename, index=False, engine='openpyxl')  # Explicitly use openpyxl

    print(f"Data saved to {filename}")

def save_queries_to_excel(filename, contexts: list[str], questions: list[str], answers: list[str], more_cols: list[tuple[str, list]]=[]):
    lists_dict = {"context": contexts, "question": questions, "answer": answers}

    for col_name, col_list in more_cols:
        lists_dict[col_name] = col_list

    save_lists_dict_to_excel(filename, lists_dict)

def drop_dataframe(dropped_dataframe: pd.DataFrame, dropping_dataframe: pd.DataFrame):
    mask = ~dropped_dataframe.set_index('id').index.isin(dropping_dataframe.set_index('id').index)

    # Filter big dropped one using the mask
    return dropped_dataframe[mask].reset_index(drop=True)

def save_dataframe(filename:str, df: pd.DataFrame):
    # Save to Excel
    df.to_excel(filename, index=False)

In [6]:
# Load and filter the dataset
full_dataset = load_dataset("taidng/UIT-ViQuAD2.0")
raw_dataset = full_dataset['train']
filtered_dataset = raw_dataset.filter(is_not_empty_answer)

In [7]:
# Convert to DataFrame
df = filtered_dataset.to_pandas()

# Assume 'context' is a column in your DataFrame
contexts = df['context'].unique()

In [8]:
query_lst = []
question_lst = []
n_samples = 0
for context in contexts:
    context_df = df[df['context'] == context]
    sampled_df = context_df.sample(n=min(n_samples_per_context, len(context_df)), random_state=42)
    sampled_questions = sampled_df['question'].tolist()
    if is_having_no_common_element(sampled_questions, question_lst):
        question_lst += sampled_questions
        query_lst.append(sampled_df)

        n_samples += 1
        if n_samples == max_samples:
            break
print(len(query_lst))
print(len(question_lst))
print(query_lst)

100
100
[               id      uit_id          title  \
1  0001-0001-0002  uit_000002  Phạm Văn Đồng   

                                             context  \
1  Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...   

                                            question  \
1  Phạm Văn Đồng giữ chức vụ gì trong bộ máy Nhà ...   

                                         answers  is_impossible  \
1  {'text': ['Thủ tướng'], 'answer_start': [60]}          False   

  plausible_answers  
1              None  ,                id      uit_id          title  \
5  0001-0002-0002  uit_000009  Phạm Văn Đồng   

                                             context  \
5  Năm 1954, ông được giao nhiệm vụ Trưởng phái đ...   

                                            question  \
5  Chức vụ mà Phạm Văn Đồng đảm nhiệm tại Hội ngh...   

                                             answers  is_impossible  \
5  {'text': ['Trưởng phái đoàn Chính phủ'], 'answ...          False   

  plausible_answers  


In [9]:
n_rows = 1000

queries = query_lst[:n_rows]
questions = get_values_in_object_lst(queries, 'question')
contexts = get_values_in_object_lst(queries, 'context')
answers = get_answer_texts_in_object_lst(queries)
print(questions)
print(contexts)
print(answers)

['Phạm Văn Đồng giữ chức vụ gì trong bộ máy Nhà nước Cộng hòa Xã hội chủ nghĩa Việt Nam?', 'Chức vụ mà Phạm Văn Đồng đảm nhiệm tại Hội nghị Genève về Đông Dương?', 'Phạm Văn Đồng từng cố gắng đưa bà Cúc đến nơi nào để chữa bệnh?', 'Tại sao việc Phạm Văn Đồng sử dụng phiên dịch người Trung Quốc tại cuộc đàm phán là sai lầm?', 'Quan điểm của Trung Quốc là gì đối với công hàm của Thủ tướng nước Việt Nam Dân Chủ Cộng Hoà Phạm Văn Đồng?', 'Lập trường của Việt Nam trong việc giải quyết các vấn đề tranh chấp quần đảo Hoàng Sa và Trường Sa là gì?', 'Tại sao lòng tin của nhân dân vào Đảng lại giảm sút?', 'Thuật ngữ thực vật hạt kín Angioosperm được định nghĩa đầu tiên trên ngôn ngữ nào?', 'Ai là người đã tìm ra những thay đổi xãy ra trong túi phôi của thực vật có hoa vào thời gian nào?', 'Nhóm thực vật nào tạo thành thảm thực vật chủ yếu trên bề mặt của Trái Đất?', 'Người ta đã phát hiện được các mẫu hoá thạch của thực vật hạt kín vào thời gian nào?', 'Thuật ngữ mang tính mô tả phổ biến nhất củ

In [10]:
save_queries_to_excel(output_file_queries_in_dataset, contexts, questions, answers)

Data saved to queries_100_uniques_in_dataset.xlsx


In [11]:
# Call ChatGPT
similar_question_responses_without_context = [generate_similar_question(question) for question in questions]

ChatGPT:  Similar question: Phạm Văn Đồng đã đảm nhiệm vai trò gì trong chính phủ Việt Nam?
ChatGPT:  Similar question: Chức vụ của Phạm Văn Đồng tại Hội nghị Genève về hòa bình là gì?
ChatGPT:  Similar question: Phạm Văn Đồng đã đưa bà Cúc đến bệnh viện nào để điều trị?
ChatGPT:  Similar question: Tại sao việc sử dụng phiên dịch người nước ngoài trong các cuộc đàm phán có thể gây ra rủi ro?
ChatGPT:  Similar question: Quan điểm của Trung Quốc về công hàm của Thủ tướng Phạm Văn Đồng trong bối cảnh quan hệ Việt-Trung là gì?
ChatGPT:  Similar question: Việt Nam có những biện pháp gì để bảo vệ chủ quyền đối với quần đảo Hoàng Sa và Trường Sa?
ChatGPT:  Similar question: Tại sao người dân lại mất niềm tin vào chính quyền?
ChatGPT:  Similar question: Thuật ngữ thực vật hạt trần Gymnosperm được định nghĩa đầu tiên trên ngôn ngữ nào?
ChatGPT:  Similar question: Ai là người phát hiện ra sự phát triển của túi phôi trong thực vật có hoa và vào thời gian nào?
ChatGPT:  Similar question: Nhóm thực

In [12]:
similar_questions_without_context = [get_answer_from_response(response, "Similar question: ") for response in similar_question_responses_without_context]
print(similar_questions_without_context)

['Phạm Văn Đồng đã đảm nhiệm vai trò gì trong chính phủ Việt Nam?', 'Chức vụ của Phạm Văn Đồng tại Hội nghị Genève về hòa bình là gì?', 'Phạm Văn Đồng đã đưa bà Cúc đến bệnh viện nào để điều trị?', 'Tại sao việc sử dụng phiên dịch người nước ngoài trong các cuộc đàm phán có thể gây ra rủi ro?', 'Quan điểm của Trung Quốc về công hàm của Thủ tướng Phạm Văn Đồng trong bối cảnh quan hệ Việt-Trung là gì?', 'Việt Nam có những biện pháp gì để bảo vệ chủ quyền đối với quần đảo Hoàng Sa và Trường Sa?', 'Tại sao người dân lại mất niềm tin vào chính quyền?', 'Thuật ngữ thực vật hạt trần Gymnosperm được định nghĩa đầu tiên trên ngôn ngữ nào?', 'Ai là người phát hiện ra sự phát triển của túi phôi trong thực vật có hoa và vào thời gian nào?', 'Nhóm thực vật nào chiếm ưu thế trong các hệ sinh thái trên Trái Đất?', 'Thực vật hạt kín xuất hiện vào thời gian nào trong lịch sử trái đất?', 'Thuật ngữ nào thường được sử dụng để chỉ thực vật có hoa?', 'Hệ thống phân loại thực vật của Cronquist được phát tri

In [13]:
original_question_without_context = ("original_question_without_context", questions)
save_queries_to_excel(output_file_queries_similar_dataset_without_context, contexts, similar_questions_without_context, answers, [original_question_without_context])

Data saved to queries_100_uniques_similar_dataset_without_context.xlsx


In [14]:
# Call ChatGPT
similar_question_responses_within_context = [generate_similar_question(question, context) for question, context in zip(questions, contexts)]

ChatGPT:  Similar question: Phạm Văn Đồng từng giữ chức vụ gì trước khi trở thành Thủ tướng của nước Cộng hòa Xã hội chủ nghĩa Việt Nam?
ChatGPT:  Similar question: Phạm Văn Đồng đã đóng vai trò gì tại Hội nghị Genève về Đông Dương?
ChatGPT:  Similar question: Phạm Văn Đồng đã đưa bà Cúc đi chữa bệnh ở những nơi nào?
ChatGPT:  Similar question: Tại sao việc Phạm Văn Đồng quá tin tưởng vào đoàn Trung Quốc trong cuộc đàm phán là một sai lầm?
ChatGPT:  Similar question: Quan điểm của Trung Quốc về công hàm của Thủ tướng Phạm Văn Đồng có ảnh hưởng gì đến tranh chấp chủ quyền ở Biển Đông?
ChatGPT:  Similar question: Lập trường của Việt Nam về việc bảo vệ chủ quyền đối với quần đảo Hoàng Sa và Trường Sa là gì?
ChatGPT:  Similar question: Tại sao một số người dân lại cảm thấy thất vọng về Đảng và Nhà nước?
ChatGPT:  Similar question: Thuật ngữ Angiosperm có nguồn gốc từ ngôn ngữ nào?
ChatGPT:  Similar question: Ai là người đã phát hiện ra sự phát triển của túi phôi trong thực vật có hoa và và

In [15]:
similar_questions_within_context = [get_answer_from_response(response, "Similar question: ") for response in similar_question_responses_within_context]
print(similar_questions_within_context)

['Phạm Văn Đồng từng giữ chức vụ gì trước khi trở thành Thủ tướng của nước Cộng hòa Xã hội chủ nghĩa Việt Nam?', 'Phạm Văn Đồng đã đóng vai trò gì tại Hội nghị Genève về Đông Dương?', 'Phạm Văn Đồng đã đưa bà Cúc đi chữa bệnh ở những nơi nào?', 'Tại sao việc Phạm Văn Đồng quá tin tưởng vào đoàn Trung Quốc trong cuộc đàm phán là một sai lầm?', 'Quan điểm của Trung Quốc về công hàm của Thủ tướng Phạm Văn Đồng có ảnh hưởng gì đến tranh chấp chủ quyền ở Biển Đông?', 'Lập trường của Việt Nam về việc bảo vệ chủ quyền đối với quần đảo Hoàng Sa và Trường Sa là gì?', 'Tại sao một số người dân lại cảm thấy thất vọng về Đảng và Nhà nước?', 'Thuật ngữ Angiosperm có nguồn gốc từ ngôn ngữ nào?', 'Ai là người đã phát hiện ra sự phát triển của túi phôi trong thực vật có hoa và vào năm nào?', 'Nhóm thực vật nào chiếm ưu thế trong các hệ sinh thái trên Trái Đất hiện nay?', 'Thực vật hạt kín xuất hiện lần đầu tiên trong khoảng thời gian nào?', 'Thuật ngữ nào được sử dụng để chỉ nhóm thực vật có hoa trong

In [16]:
original_question_within_context = ("original_question_within_context", questions)
original_context_within_context = ("original_context_within_context", contexts)
save_queries_to_excel(output_file_queries_similar_dataset_within_context, contexts, similar_questions_within_context, answers, [original_question_within_context, original_context_within_context])

Data saved to queries_100_uniques_similar_dataset_within_context.xlsx


In [17]:
# Dataset to DataFrame
filtered_df = pd.DataFrame(filtered_dataset)

# list to DataFrame
query_df = pd.concat(query_lst[:n_rows])

dropped_df = drop_dataframe(filtered_df, query_df)
print(len(filtered_df))
print(len(query_df))
print(len(dropped_df))

19238
100
19138


In [18]:
save_dataframe(output_file_dataset_filterd_full, filtered_df)

In [19]:
save_dataframe(output_file_dataset_filterd_dropped, dropped_df)